# Presentation 01 — Dataset overview

Figures describing **what data the model was trained on**: per-species sample counts, total audio duration, and one example waveform + mel-spectrogram per class.

Run cells in order. All generated figures are saved to `outputs/presentation/` on your Drive so you can drop them straight into slides.

## Setup — mount Drive, clone latest code, configure paths

This block is the same across all presentation notebooks. It mounts your Drive, pulls the latest branch, adds `src/` to `sys.path` and sets `PRIMATE_DATA_ROOT`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf primates-sound-detection
!git clone -q -b claude/review-repo-QAc6j https://github.com/Mo119m/primates-sound-detection.git
%cd /content/primates-sound-detection
!pip install -q -r requirements.txt

In [ ]:
import os, sys, importlib
PROJECT_ROOT = '/content/primates-sound-detection'
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ['PRIMATE_DATA_ROOT'] = '/content/drive/MyDrive/primates-data'

import config
importlib.reload(config)

PRESENT_DIR = os.path.join(config.OUTPUT_ROOT, 'presentation')
os.makedirs(PRESENT_DIR, exist_ok=True)
print('Presentation figures will be saved to:', PRESENT_DIR)

In [ ]:
# Consistent matplotlib style for every figure in this notebook
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 220,
    'savefig.bbox': 'tight',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.frameon': False,
})

# Shared species color palette
SPECIES_COLORS = {
    'Cercopithecus_nictitans': '#E67E22',  # orange
    'Colobus_guereza':         '#27AE60',  # green
    'Pan_troglodytes':         '#2980B9',  # blue
    'Background':              '#7F8C8D',  # grey
}

def save_fig(fig, name):
    path = os.path.join(PRESENT_DIR, name)
    fig.savefig(path)
    print(f'  saved -> {path}')
    return path

## Load raw audio

We load every training clip (species + background) off Drive. This is slow the first time because of Drive I/O — expect a few minutes.

In [ ]:
import data_loader
importlib.reload(data_loader)

species_data = data_loader.load_species_data()
background_data = data_loader.load_background_data()
data_loader.print_data_summary(species_data, background_data)

## Figure 1 — Number of training clips per class

Shows how balanced (or imbalanced) the raw dataset is before augmentation.

In [ ]:
import numpy as np

counts = {sp: len(v) for sp, v in species_data.items()}
counts['Background'] = len(background_data)

classes = list(counts.keys())
values  = [counts[c] for c in classes]
colors  = [SPECIES_COLORS.get(c, '#888') for c in classes]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(classes, values, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Number of 5 s clips')
ax.set_title('Training clips per class')
ax.set_xticklabels(classes, rotation=20, ha='right')
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v, str(v),
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.margins(y=0.15)
plt.tight_layout()
save_fig(fig, '01_clips_per_class.png')
plt.show()

## Figure 2 — Total audio duration per class

Same story but in minutes of audio (each clip is 5 s). Useful when you want to talk about how many **hours** of labelled data the model saw.

In [ ]:
minutes = {c: (v * 5.0 / 60.0) for c, v in counts.items()}

classes = list(minutes.keys())
values  = [minutes[c] for c in classes]
colors  = [SPECIES_COLORS.get(c, '#888') for c in classes]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(classes, values, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Audio (minutes)')
ax.set_title('Total labelled audio per class')
ax.set_xticklabels(classes, rotation=20, ha='right')
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:.1f} min',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.margins(y=0.15)
plt.tight_layout()
save_fig(fig, '02_minutes_per_class.png')
plt.show()

total_min = sum(minutes.values())
print(f'\nTotal labelled audio: {total_min:.1f} min ({total_min/60:.1f} h)')

## Figure 3 — One example mel-spectrogram per species

Picks the **most energetic** clip per species (highest RMS) and plots the mel-spectrogram. We avoid random sampling because some clips in the dataset are shorter than 5 s and get zero-padded — a random draw can hit one of those and produce a mostly-black spectrogram. Picking by RMS guarantees a clip with real signal across most of its duration.

In [ ]:
import preprocessing
importlib.reload(preprocessing)
import librosa
import librosa.display

SR = config.SAMPLE_RATE  # data_loader resamples everything to this

def pick_best_example(audio_list):
    '''Pick the clip with the highest RMS — avoids zero-padded silent clips.'''
    best_idx, best_rms = 0, -1.0
    for i, item in enumerate(audio_list):
        audio = item[0]
        rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
        if rms > best_rms:
            best_rms, best_idx = rms, i
    return best_idx

# data_loader returns (audio, file_path) tuples, NOT (audio, sr)
example_items = list(species_data.items())
example_items.append(('Background', background_data))

# Precompute the chosen indices so Figures 3 and 4 use the same clips
chosen = {name: pick_best_example(audio_list) if len(audio_list) else None
          for name, audio_list in example_items}

n = len(example_items)
fig, axes = plt.subplots(n, 1, figsize=(11, 3.2 * n))
if n == 1:
    axes = [axes]

for ax, (name, audio_list) in zip(axes, example_items):
    if chosen[name] is None:
        ax.set_title(f'{name} (no samples)')
        continue
    audio = audio_list[chosen[name]][0]
    mel_spec = preprocessing.audio_to_melspectrogram(audio, SR)
    img = librosa.display.specshow(
        mel_spec, sr=SR, hop_length=config.HOP_LENGTH,
        x_axis='time', y_axis='mel', fmin=config.FMIN, fmax=config.FMAX,
        ax=ax, cmap='magma',
    )
    ax.set_title(f'{name}', color=SPECIES_COLORS.get(name, '#333'), fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Mel freq (Hz)')
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
save_fig(fig, '03_example_spectrograms.png')
plt.show()

## Figure 4 — Waveforms side-by-side

The same (RMS-picked) clips as Figure 3 but as raw waveforms. Makes it very clear visually how different the three species calls are in amplitude / rhythm.

In [ ]:
fig, axes = plt.subplots(n, 1, figsize=(11, 2.2 * n))
if n == 1:
    axes = [axes]

for ax, (name, audio_list) in zip(axes, example_items):
    if chosen[name] is None:
        ax.set_title(f'{name} (no samples)')
        continue
    audio = audio_list[chosen[name]][0]
    t = np.arange(len(audio)) / SR
    ax.plot(t, audio, linewidth=0.6, color=SPECIES_COLORS.get(name, '#333'))
    ax.set_title(name, color=SPECIES_COLORS.get(name, '#333'), fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amp')
    ax.set_xlim(0, t[-1] if len(t) > 0 else 5.0)

plt.tight_layout()
save_fig(fig, '04_example_waveforms.png')
plt.show()

## Figure 5 — Summary table (for an appendix slide)

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'class': classes,
    'n_clips': [counts[c] for c in classes],
    'minutes': [round(minutes[c], 1) for c in classes],
})
summary['minutes'] = summary['minutes'].astype(float)
summary.loc['TOTAL'] = ['TOTAL', summary['n_clips'].sum(), round(summary['minutes'].sum(), 1)]
print(summary.to_string(index=False))

summary.to_csv(os.path.join(PRESENT_DIR, '05_dataset_summary.csv'), index=False)
print(f'\n  saved -> {os.path.join(PRESENT_DIR, "05_dataset_summary.csv")}')

## Done

Figures `01_*.png` through `04_*.png` and `05_dataset_summary.csv` are now on your Drive under `outputs/presentation/`. Download and drop them into your slide deck.